# Preprocessing functions

In [1]:
import matplotlib as plt
import cv2
import numpy as np

## Denoise

In [2]:
def denoise_image(image, kernel_size=(5, 5), sigma=0, show=False):
    if show:
        denoised = cv2.GaussianBlur(image, kernel_size, sigma)
        plt.imshow(denoised, cmap='gray')
        plt.title('Denoised Image')
        plt.axis('off')
        plt.show()
        return denoised
    else: 
        return cv2.GaussianBlur(image, kernel_size, sigma)

## Binarization

In [3]:
# Função de binarização (thresholding)
def binarize_image(image, threshold=127, max_value=255, method=cv2.THRESH_BINARY, show=False):
    _, binary = cv2.threshold(image, threshold, max_value, method)
    if show:
        plt.imshow(binary, cmap='gray')
        plt.title('Binarized Image')
        plt.axis('off')
        plt.show()
    return binary

## Lowpass filter

In [4]:
def lowpass_filter(image, kernel_size=(5, 5), show=False):
    blurred = cv2.blur(image, kernel_size)
    if show:
        plt.imshow(blurred, cmap='gray')
        plt.title('Low-pass Filtered Image')
        plt.axis('off')
        plt.show()
    return blurred

## Morphological operations

In [5]:
def erode_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    eroded = cv2.erode(image, kernel, iterations=iterations)
    if show:
        plt.imshow(eroded, cmap='gray')
        plt.title('Eroded Image')
        plt.axis('off')
        plt.show()
    return eroded

def dilate_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    dilated = cv2.dilate(image, kernel, iterations=iterations)
    if show:
        plt.imshow(dilated, cmap='gray')
        plt.title('Dilated Image')
        plt.axis('off')
        plt.show()
    return dilated

def open_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    opened = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel, iterations=iterations)
    if show:
        plt.imshow(opened, cmap='gray')
        plt.title('Opened Image')
        plt.axis('off')
        plt.show()
    return opened

def close_image(image, kernel_size=(3, 3), iterations=1, show=False):
    kernel = np.ones(kernel_size, np.uint8)
    closed = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    if show:
        plt.imshow(closed, cmap='gray')
        plt.title('Closed Image')
        plt.axis('off')
        plt.show()
    return closed

## Define preprocessing functions into a mapping

In [6]:
# definning wrappers for each preprocessing operation
def identity(img):
    return img

def denoise_wrapper(img):
    return denoise_image(img)

def binarize_wrapper(img):
    return binarize_image(img)

def lowpass_wrapper(img):
    return lowpass_filter(img)

def erode_wrapper(img):
    return erode_image(img)

def dilate_wrapper(img):
    return dilate_image(img)

def open_wrapper(img):
    return open_image(img)

def close_wrapper(img):
    return close_image(img)

preprocessing_methods = {
    'none': identity,
    'denoise': denoise_wrapper,
    'binarize': binarize_wrapper,
    'lowpass': lowpass_wrapper,
    'erode': erode_wrapper,
    'dilate': dilate_wrapper,
    'open': open_wrapper,
    'close': close_wrapper
}

# Preprocessing and Running the Model Workflow

We will now apply each preprocessing operation (denoise, binarization, lowpass filter, morphological operations), as well as the mixing of these operations, to the dataset, then train the Keras model on each preprocessed version, and compare their validation accuracy to determine the best approach for the problem.

In [7]:
def preprocess_images(df, preproc_fn):
    X = []
    y = []
    for idx, row in df.iterrows():
        img_path = row['image_path']
        label = row['label']
        
        if 7 <= label <= 14: # remap labels 7-14 to 0-7
            label = label - 7
        else:
            continue  # skip labels outside 7-14
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
        processed = preproc_fn(img)
        X.append(processed)
        y.append(label)
    X = np.stack(X)
    y = np.array(y)
    return X, y

## Building the Model and Running the Model

In [8]:
import pandas as pd

train_df = pd.read_csv('train_split.csv')
test_df = pd.read_csv('test_split.csv')

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 1968 samples
Test set: 492 samples


In [9]:
def build_model(input_shape, num_classes):
    from keras.models import Sequential
    from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
from itertools import product
import numpy as np
import pandas as pd

results = []
preproc_names = list(preprocessing_methods.keys())

for name in preproc_names:
    if name == 'none':
        continue
    print(f"\n=== Preprocessing: {name} ===")
    X_train, y_train = preprocess_images(train_df, preprocessing_methods[name])
    X_test, y_test = preprocess_images(test_df, preprocessing_methods[name])
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    model = build_model(input_shape=X_train.shape[1:], num_classes=8)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=10,
        batch_size=32,
        verbose=2
    )
    val_accuracies = history.history['val_accuracy']
    best_epoch = int(np.argmax(val_accuracies)) + 1
    best_val_acc = float(np.max(val_accuracies))
    results.append({'preprocessing': name, 'best_val_accuracy': best_val_acc, 'best_epoch': best_epoch})
    print(f"Best val accuracy for {name}: {best_val_acc:.4f} at epoch {best_epoch}")

# Mixed methods (all pairs, both orders, skip identity-identity)
for first, second in product(preproc_names, repeat=2):
    if first == 'none' and second == 'none':
        continue  # skip identity-identity
    def mixed_fn(img, f=first, s=second):
        return preprocessing_methods[s](preprocessing_methods[f](img))
    print(f"\n=== Preprocessing: {first} -> {second} ===")
    X_train, y_train = preprocess_images(train_df, mixed_fn)
    X_test, y_test = preprocess_images(test_df, mixed_fn)
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    model = build_model(input_shape=X_train.shape[1:], num_classes=8)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=10,
        batch_size=32,
        verbose=2
    )
    val_accuracies = history.history['val_accuracy']
    best_epoch = int(np.argmax(val_accuracies)) + 1
    best_val_acc = float(np.max(val_accuracies))
    results.append({'preprocessing': f'{first}->{second}', 'best_val_accuracy': best_val_acc, 'best_epoch': best_epoch})
    print(f"Best val accuracy for {first}->{second}: {best_val_acc:.4f} at epoch {best_epoch}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_val_accuracy', ascending=False)
results_df


=== Preprocessing: denoise ===
